# Graph and Harmonic Centrality Algorithm

We want to evaluate if our current distribution point (kitchen) is in the center of the graph, and if we should potentially move it. We will run the harmonic centrality algorithm to find the center or the top central stations in the graph. This will allow us to see how far off our distribution point is from the center of the graph, whether we should move it, and potential candidates for moving it.

# Import libraries

In [1]:
import neo4j
from IPython.display import display

import csv

import math
import numpy as np
import pandas as pd

import psycopg2


# Supporting code

Code cells containing any supporting code, such as connecting to the database, any functions, etc.  

In [2]:
def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)

In [3]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))

In [4]:
session = driver.session(database="neo4j")

In [5]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

In [6]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [7]:
def my_neo4j_number_nodes_relationships():
    "print the number of nodes and relationships"
   
    
    query = """
        match (n) 
        return n.name as node_name, labels(n) as labels
        order by n.name
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_nodes = df.shape[0]
    
    
    query = """
        match (n1)-[r]->(n2) 
        return n1.name as node_name_1, labels(n1) as node_1_labels, 
            type(r) as relationship_type, n2.name as node_name_2, labels(n2) as node_2_labels
        order by node_name_1, node_name_2
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_relationships = df.shape[0]
    
    print("-------------------------")
    print("  Nodes:", number_nodes)
    print("  Relationships:", number_relationships)
    print("-------------------------")


In [8]:
def my_neo4j_create_node(station_name):
    "create a node with label Station"
    
    query = """
    
    CREATE (:Station {name: $station_name})
    
    """
    
    session.run(query, station_name=station_name)

In [9]:
def my_neo4j_create_relationship_one_way(from_station, to_station, weight):
    "create a relationship one way between two stations with a weight"
    
    query = """
    
    MATCH (from:Station), 
          (to:Station)
    WHERE from.name = $from_station and to.name = $to_station
    CREATE (from)-[:LINK {weight: $weight}]->(to)
    
    """
    
    session.run(query, from_station=from_station, to_station=to_station, weight=weight)
    

In [10]:
def my_neo4j_create_relationship_two_way(from_station, to_station, weight):
    "create relationships two way between two stations with a weight"
    
    query = """
    
    MATCH (from:Station), 
          (to:Station)
    WHERE from.name = $from_station and to.name = $to_station
    CREATE (from)-[:LINK {weight: $weight}]->(to),
           (to)-[:LINK {weight: $weight}]->(from)
    
    """
    
    session.run(query, from_station=from_station, to_station=to_station, weight=weight)
    

In [11]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [12]:
cursor = connection.cursor()

# Drop Tables

In [13]:
connection.rollback()

query = """

drop table if exists stations;
drop table if exists lines;
drop table if exists travel_times;

"""

cursor.execute(query)

connection.commit()


# Create Tables

In [14]:
connection.rollback()

query = """

create table stations (
  station varchar(32),
  latitude numeric(9,6),
  longitude numeric(9,6),
  transfer_time numeric(3),
  primary key (station)
);

create table lines (
  line varchar(6),
  sequence numeric(2),
  station varchar(32),
  primary key (line, sequence)
);

create table travel_times (
  station_1 varchar(32),
  station_2 varchar(32),
  travel_time numeric(3),
  primary key (station_1, station_2)
);

"""

cursor.execute(query)

connection.commit()

# Load Data into Database Tables

In [15]:
connection.rollback()

query = """

copy stations
from '/user/projects/project-3-AppleTater/exercise/stations.csv' delimiter ',' NULL '' csv header;

copy lines
from '/user/projects/project-3-AppleTater/exercise/lines.csv' delimiter ',' NULL '' csv header;

copy travel_times
from '/user/projects/project-3-AppleTater/exercise/travel_times.csv' delimiter ',' NULL '' csv header;

"""

cursor.execute(query)

connection.commit()

# Verify Data Loaded Correctly into Database Tables

In [16]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from stations
order by station

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,station,latitude,longitude,transfer_time
0,12th Street,37.803608,-122.272006,282
1,16th Street Mission,37.764847,-122.420042,287
2,19th Street,37.807869,-122.268980,67
3,24th Street Mission,37.752000,-122.418700,277
4,Antioch,37.996281,-121.783404,0
5,Ashby,37.853068,-122.269957,299
6,Balboa Park,37.721667,-122.447500,48
7,Bay Fair,37.697000,-122.126500,63
8,Berryessa,37.368361,-121.874655,288
9,Castro Valley,37.690748,-122.075679,0


In [17]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from lines
order by line,sequence

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,line,sequence,station
0,blue,1,Dublin
1,blue,2,West Dublin
2,blue,3,Castro Valley
3,blue,4,Bay Fair
4,blue,5,San Leandro
...,...,...,...
109,yellow,23,Daly City
110,yellow,24,Colma
111,yellow,25,South San Francisco
112,yellow,26,San Bruno


In [18]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from travel_times
order by station_1,station_2

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,station_1,station_2,travel_time
0,12th Street,19th Street,120
1,12th Street,Lake Merritt,180
2,12th Street,West Oakland,300
3,16th Street Mission,24th Street Mission,120
4,16th Street Mission,Civic Center,180
5,19th Street,MacArthur,180
6,24th Street Mission,Glen Park,180
7,Antioch,Pittsburg Center,420
8,Ashby,Downtown Berkeley,180
9,Ashby,MacArthur,240


# Create Graph

Create a graph database in Neo4j for the BART sytem. 

We will use the functions created above to create the nodes and relationships:
* my_neo4j_create_node() - creates a node with label Station
* my_neo4j_create_relationship_one_way() - creates a relationship one way between two stations with a weight
* my_neo4j_create_relationship_two_way() - create relationships two way between two stations with a weight


## Wipe out the Neo4j database

Call the function my_neo4j_wipe_out_database() to wipe out the Neo4j database

In [19]:
my_neo4j_wipe_out_database()

## Verify the number of nodes and relationships
Call the function my_neo4j_number_nodes_relationships()

In [20]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 0
  Relationships: 0
-------------------------


## Query the list of stations and create the departure and arrival nodes in the graph

For each station X, create two nodes:
* depart X
* arrive X

Use the function my_neo4j_create_node() defined above

For example, West Oakland:
* my_neo4j_create_node('depart West Oakland')
* my_neo4j_create_node('arrive West Oakland')


In [21]:
connection.rollback()

query = """

select station
from stations
order by station

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    station = row[0]
    
    my_neo4j_create_node('depart ' + station)
    my_neo4j_create_node('arrive ' + station)

## Verify the number of nodes and relationships

Call the function my_neo4j_number_nodes_relationships() 

In [22]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 100
  Relationships: 0
-------------------------


## Query the list of stations and the lines they serve, create line nodes, and create relationships between the line nodes and the departure and arrival nodes with weight 0

For each station X and each line Y that the station serves:
* Create a line node
* Create a relationship from the departure node to the line node with weight 0
* Create a relationship from the line node to the arrival node with weight 0

Use the function my_neo4j_create_relationship_one_way() defined above to create the relationships

In [23]:
connection.rollback()

query = """

select station, line
from lines
order by station, line

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    station = row[0]
    line = row[1]
    
    depart = 'depart ' + station
    arrive = 'arrive ' + station
    line_station = line + ' ' + station
    
    my_neo4j_create_node(line_station)
    my_neo4j_create_relationship_one_way(depart, line_station, 0)
    my_neo4j_create_relationship_one_way(line_station, arrive, 0)

## Verify the number of nodes and relationships

Call the function my_neo4j_number_nodes_relationships() 

In [24]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 214
  Relationships: 228
-------------------------


## Query the list of all possible line transfers and the transfer times, create a relationship for each transfer with the transfer time as the weight

For each station X, from line Y, to line Z, create a relationship from Y's line node to Z's line node with the weight set to the transfer time

In [25]:
connection.rollback()

query = """

select a.station, a.line as from_line, b.line as to_line, s.transfer_time
from lines a
     join lines b
       on a.station = b.station and a.line <> b.line 
     join stations s
       on a.station = s.station
order by 1, 2, 3

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    station = row[0]
    from_line = row[1]
    to_line = row[2]
    transfer_time = int(row[3])
    
    from_station = from_line + ' ' + station
    to_station = to_line + ' ' + station
    
    my_neo4j_create_relationship_one_way(from_station, to_station, transfer_time)

## Verify the number of nodes and relationships
Call the function my_neo4j_number_nodes_relationships() 

In [26]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 214
  Relationships: 436
-------------------------


## Query the list of all segments between each station and its adjoining stations, create a relationship for each segment both ways

For each segment from station X to station Y on line Z, create two relationships:
* From X's line node to Y's line node with travel time
* From Y's line node to X's line node with travel time

Use the function my_neo4j_create_relationship_two_way() defined above which will create both relationships 

In [27]:
connection.rollback()

query = """

select a.line, a.station as from_station, b.station as to_station, t.travel_time
from lines a
  join lines b
    on a.line = b.line and b.sequence = (a.sequence + 1)
  join travel_times t
    on (a.station = t.station_1 and b.station = t.station_2)
        or (a.station = t.station_2 and b.station = t.station_1)
order by line, from_station, to_station

"""

cursor.execute(query)

connection.rollback()

rows = cursor.fetchall()

for row in rows:
    
    line = row[0]
    from_station = line + ' ' + row[1]
    to_station = line + ' ' + row[2]
    travel_time = int(row[3])
    
    my_neo4j_create_relationship_two_way(from_station, to_station, travel_time)

## Verify the number of nodes and relationships

Call the function my_neo4j_number_nodes_relationships()

In [28]:
my_neo4j_number_nodes_relationships()

-------------------------
  Nodes: 214
  Relationships: 652
-------------------------


# Harmonic Centrality Algorithm

In [29]:
query = "CALL gds.graph.drop('ds_graph', false)"
session.run(query)

query = "CALL gds.graph.project('ds_graph', 'Station', 'LINK', {relationshipProperties: 'weight'})"
session.run(query)

In [30]:
query = """

CALL gds.alpha.closeness.harmonic.stream('ds_graph', {})
YIELD nodeId, centrality
RETURN gds.util.asNode(nodeId).name AS name, centrality as closeness
ORDER BY centrality DESC

"""

line_nodes = my_neo4j_run_query_pandas(query)
line_nodes

,name,closeness
0,green West Oakland,0.160198
1,yellow West Oakland,0.159922
2,blue West Oakland,0.159903
3,red West Oakland,0.159674
4,green Embarcadero,0.154619
...,...,...
209,depart Union City,0.000000
210,depart Walnut Creek,0.000000
211,depart Warm Springs,0.000000
212,depart West Dublin,0.000000


# Consolidate Line Nodes per Station

In [39]:
# For the line nodes, such as, "red X", "green X", etc. consolidate them into just an X value by 
# taking the average of the line node values.

# Get 'name' and 'closeness' as lists
station_names = []
closeness = []
for ind in line_nodes.index:
    station_names.append(line_nodes['name'][ind])
    closeness.append(line_nodes['closeness'][ind])
    
# Remove tracks (e.g. colors) from names of stations
for i in range(len(station_names)):
    # ignore depart X, arrive X
    ignore_node = ' '.join (station_names[i].split(" ")[0])
    if (ignore_node not in ['depart', 'arrive']):
        station_names[i] = ' '.join (station_names[i].split(" ")[1:] ) # split, remove tracks, convert list back to string
    else:
        # remove index from closeness
        del closeness[index]

# Remove duplicates from list of station names
stations_set = list(set(station_names))

# Create dictionary with stations as keys and list of closeness as values
stations_dict = dict([(k, []) for k in stations_set])
# Add closeness to dictionary
for key in stations_dict.keys():
    for j in range(len(closeness)):
        if key == station_names[j]:
            stations_dict[key].append(closeness[j])

# Calculate average per station
for key in stations_dict.keys():
    stations_dict[key] = sum(stations_dict[key])/ len(stations_dict[key])
                        
# Convert dictionary of stations to a pd
stations_pd = pd.DataFrame(list(stations_dict.items()), columns=['name', 'closeness'])
stations_pd.sort_values(by='closeness', ascending=False)

,name,closeness
6,West Oakland,0.131206
21,Embarcadero,0.126800
28,Montgomery Street,0.123487
26,Powell Street,0.120650
45,Lake Merritt,0.120233
30,12th Street,0.120186
33,Civic Center,0.117977
2,Coliseum,0.117078
46,16th Street Mission,0.115257
47,Fruitvale,0.115005
